In [ ]:
import pandas as pd

dfDef = pd.read_csv("../resources/gdpdef.csv")
dfDef.head()
dfDef['Year'] = dfDef['date'].str.split('-').str[0].astype(int)
arrDef = dfDef.set_index('Year')['gdpdef']
arrDef[2024] = (139.71495 / 136.41462) * arrDef[2023]
def gcam_deflator(value, from_year=2023, to_year=1975):
    return value * arrDef[to_year] / arrDef[from_year]

dfExc = pd.read_csv("../resources/DEXKOUS.csv")
dfExc['date'] = pd.to_datetime(dfExc['observation_date'])
dfExcAnnual = dfExc.groupby(dfExc['date'].dt.year)['DEXKOUS'].mean()

def krw_to_usd(year):
    return 1 / dfExcAnnual[year]

* Data Source:
    * [European Environment Agency](https://www.eea.europa.eu/en/analysis/indicators/use-of-auctioning-revenues-generated)
    * Greenhouse Gas Inventory and Research Center of Korea (GIR), Ministry of Environment, 2020 Report on the Operation of the Emissions Trading Scheme (ROETS, `../resources/ROETS-2022-GIR`)
    * [RE100 Information Platform](https://www.k-re100.or.kr/doc/sub2_4_1.php)

* Implemented Input Files
    * `/input/policy/korea-2035/industry/carbon_tax_cp.xml`
    * `/input/policy/korea-2035/industry/carbon_tax_ep.xml`
    * `/input/policy/korea-2035/industry/CO2_CBAM.xml`

# Carbon Prices under the Korean Emissions Trading Scheme

As of June, 2025, the carbon price under the Emissions Trading Scheme in Korea remains 8,870 KRW / tCO2.

According to [GIR](https://www.gir.go.kr/home/board/read.do;jsessionid=k7vRM8kUakHQIjVwFuxyyxqmYmxWLZEyYAWi2cURy6eaCfGTzrp2RH6OBFtxuscV.og_was2_servlet_engine1?pagerOffset=30&maxPageItems=10&maxIndexPages=10&searchKey=&searchValue=&menuId=10&boardMasterId=4&boardId=165),  the long-term average carbon price in 2020 was 18,510 KRW / tCO2. Also, the short-tem average carbon price in 2020 was 30,411 KRW / tCO2, says [another source](https://www.ohmynews.com/NWS_Web/View/img_pg.aspx?CNTN_CD=IE003466107).

In [26]:
tco2_2020_krw_2020 = 18510
tco2_2020_usd_2020 = tco2_2020_krw_2020 * krw_to_usd(2020)
tC_2020_usd_1990 = tco2_2020_usd_2020 * gcam_deflator(1, 2020, 1990) * 3.666667
print(tC_2020_usd_1990)

32.35958387462024


In [27]:
tco2_2020_krw_2020 = 30411
tco2_2020_usd_2020 = tco2_2020_krw_2020 * krw_to_usd(2020)
tC_2020_usd_1990 = tco2_2020_usd_2020 * gcam_deflator(1, 2020, 1990) * 3.666667
print(tC_2020_usd_1990)

53.165170459809616


# CBAM

In 2023, the average carbon price under the EU Emissions Trading System (EU ETS) was 83 € / tCO2. Applying Hotelling’s rule with an annual growth rate of 2.5%, we project future carbon prices for the EU ETS.

In [28]:
83 * 1.025 ** 5

93.90688166992183

In [29]:
83 * 1.025 ** 10

106.2470171682976

In [30]:
tax_tCO2_eur_25 = 83
growth_rate = 0.025
tax_tCO2_eur_30 = tax_tCO2_eur_25 * (1+growth_rate) ** 5
tax_tCO2_eur_35 = tax_tCO2_eur_30 * (1+growth_rate) ** 5
exchange_rate_2023 = 0.9241
tax_tCO2_usd_30 = tax_tCO2_eur_30 / exchange_rate_2023
tax_tCO2_usd_35 = tax_tCO2_eur_35 / exchange_rate_2023

In [31]:
dictTax = {}
dictTax[2030] = gcam_deflator(tax_tCO2_usd_30, 2023, 1990) * 3.666667
dictTax[2035] = gcam_deflator(tax_tCO2_usd_35, 2023, 1990) * 3.666667
dictTax

{2030: np.float64(180.7218490423591), 2035: np.float64(204.47018425530476)}

We apply an export share of 13.5% for iron and steel products and 15% for chemical products to calculate the CBAM certificate purchase cost as

CBAM Certificate Purchase Cost = (EU ETS Carbon Price − K ETS Carbon Price) × Export Share